# 04 · Final runs — GPU models (XGBoost, MLP, GraphSAGE, GATv2, ChebNet, GIN)

This notebook trains the final models of the project: every (menu, model) pair tuned in step 03 is trained again **from scratch with three new seeds** (0, 1, 2), with the hyper-parameters chosen by the tuning, and evaluated for the first time on the **test set** (time steps 41–49). Its twin `04-final-runs-cpu.ipynb` does the same for Logistic Regression and Random Forest. Both write into the same folder.

**Runtime: GPU** (`Runtime → Change runtime type → T4 GPU`). Expected time: about **1.5 hours** (GATv2 is about half of it), usually one Colab session.

| Model | Menus | Triples | Device |
|---|---|---|---|
| XGBoost | the 6 menus with features | 18 | GPU |
| MLP | the 6 menus with features | 18 | GPU |
| GraphSAGE | the 6 menus with features | 18 | GPU |
| GATv2 | the 6 menus with features | 18 | GPU |
| ChebNet | the 6 menus with features + `none` | 21 | GPU |
| GIN | `none` | 3 | GPU |

That is 32 pairs × 3 seeds = **96 triples**.

**How a triple is trained.** `set_seed(seed)` (Python, NumPy, PyTorch, cuDNN), then the model is built and trained exactly as in the tuning, with the hyper-parameters of `03-tuning/best/<menu>__<model>.json`:
- **XGBoost**: up to 2,000 trees, early stopping after 100 trees without improvement of the validation PR-AUC, `random_state = seed`;
- **MLP and GNNs**: full batch, Adam, binary cross-entropy weighted by $w_+ = \#\text{licit} / \#\text{illicit} \approx 8.06$; early stopping after 50 (MLP) or 100 (GNN) epochs without improvement of the validation PR-AUC, at most 1,000 epochs; the saved network is the one of the **best epoch**. The GNNs train on the graph of time steps 1–30 (unknown nodes included) and score the graphs of time steps 31–40 and 41–49. Every 10 epochs the PR-AUC of the train is also computed, for the overfitting curves.

The validation set chooses the early-stopping point and the F1 threshold; the test set is only scored.

**Metrics** (illicit = positive class). Every final run saves the score $s \in [0, 1]$ of every evaluation row, so any metric can be recomputed later; the summary already has these, for train, validation and test:
- **PR-AUC** (average precision), the main metric: $\mathrm{AP} = \sum_k (R_k - R_{k-1})\,P_k$, where $P_k$ and $R_k$ are precision and recall when the transactions with the $k$ highest distinct scores are flagged. A random classifier gets the prevalence of the set: 0.110 on train, 0.113 on validation, 0.053 on test. Because the test has half the prevalence, its PR-AUC is lower even without overfitting;
- **ROC-AUC**: the probability that a random illicit transaction has a higher score than a random licit one (ties count one half). It does not depend on the prevalence, so it compares validation and test fairly;
- at **two thresholds** (a transaction is flagged when $s \ge t$): $t = 0.5$, and the threshold with the **best illicit F1 on validation**, applied unchanged to the test (choosing it on the test would make the test F1 optimistic). At each threshold: $tp, fp, tn, fn$, and from them
$$P = \frac{tp}{tp+fp}, \quad R = \frac{tp}{tp+fn}, \quad F_1 = \frac{2\,tp}{2\,tp+fp+fn}, \quad \mathrm{MCC} = \frac{tp\cdot tn - fp\cdot fn}{\sqrt{(tp+fp)(tp+fn)(tn+fp)(tn+fn)}},$$
for the illicit class, the same for the licit class (swap the roles), and the macro F1 = the mean of the two F1. The illicit F1 is the measure used by Weber et al. (2019) on this data;
- **top-k on the test**: sort the test transactions by score and look at the first $k$, as an analyst who can check only $k$ of them. precision@k = illicit among the $k$ / $k$; recall@k = illicit among the $k$ / all 524 illicit. $k$ = 10, 99 and 987 (0.1%, 1% and 10% of the test), 1,090 (the train prevalence times the test size), 100, and 524 (where precision and recall are equal);
- **bootstrap on the test**: 1,000 resamples of the 9,866 test rows with replacement; the metric on each one gives its spread (standard deviation and the 2.5–97.5 percentile interval). The resamples come from a fixed seed (0) and are **the same for every triple**, so two models can be compared resample by resample. The 1,000 values of each triple are saved.

The spread over the three seeds (retraining) and the bootstrap spread (another sample of test transactions) are two different uncertainties: both are reported.

**Output** (in `DATA_PATH/04-final-runs/`, shared with the other final-run notebook). One file per triple in each folder, named `<menu>__<model>__seed<s>`:

| File | Content |
|---|---|
| `predictions/<triple>.parquet` | one row per evaluation row of train, validation and test: `row` (node id), `txId`, `time_step`, `split`, `y`, `score`, and for the networks `logit` (the score before the sigmoid). Parquet keeps the exact scores (a CSV with 10 digits would turn the most confident scores into ties, which change the PR-AUC) |
| `summary/<triple>.json` | hyper-parameters, training information (time, best epoch or iteration, memory, parameters), every metric above, library versions, code version. Written last: its presence means "triple finished" |
| `curves/<triple>.csv` | networks: one row per epoch (train and validation loss, validation PR-AUC, train PR-AUC every 10 epochs); XGBoost: one row per tree (validation PR-AUC computed by XGBoost; train and validation PR-AUC every 10 trees) |
| `importances/<triple>.csv` | XGBoost only: the gain of every feature (the networks are explained later, from the saved weights) |
| `bootstrap/<triple>.csv` | the 1,000 bootstrap values of test PR-AUC, ROC-AUC and illicit F1 |
| `models/<triple>.json | pt` | the trained model: XGBoost in its own JSON format, the networks as PyTorch `state_dict` (the weights of the best epoch; the summary has everything needed to rebuild the network). Embeddings are not saved: step 05 recomputes them from the weights |
| `determinism-gpu.csv` | the reproducibility check (section 4) |
| `logs/04-final-runs-gpu.log` | the log lines of this notebook |
| `runs.csv` | one row per finished triple, of both final-run notebooks |

No plots here: they are all made in step 05, from these files.

**Resume.** A triple is skipped when its `summary` file exists. If Colab disconnects, run the notebook again from the top: at most the triple that was running is lost. The files are written through a temporary file and renamed, so an interrupted write never leaves a half-written file.

**Shared code.** The cells marked `SHARED CODE` are identical copies of the cells of the tuning notebooks: data preparation, transformations, models and training loop are exactly those of the tuning. The notebook stops if the files of step 03 were written by another version of that code (`SHARED_CODE_VERSION`). The cell `FINAL-RUN CODE` is the same in both final-run notebooks.

## 0. Setup

In [ ]:
# Install the libraries that Colab does not have by default (only if they are missing).
import importlib.util
import subprocess
import sys

for package, module in [('torch_geometric', 'torch_geometric')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

In [ ]:
# Connect Google Drive: every input and output of the project lives in DATA_PATH.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/UNIVERSITA/block-crypto/data')
INPUT_DIR = DATA_PATH / '02-feature-menus'      # output of step 02
TUNING_DIR = DATA_PATH / '03-tuning'            # output of step 03 (best hyper-parameters)
RUNS_DIR = DATA_PATH / '04-final-runs'          # output of the final-run notebooks
LOCAL_INPUT = Path('/content/input')            # local copy of the input files

In [ ]:
# Copy the inputs from Drive to the Colab disk: reading a local file is much faster.
import shutil

LOCAL_INPUT.mkdir(exist_ok=True)
for name in ['config.json', 'dataset.parquet', 'edges.parquet']:
    source, target = INPUT_DIR / name, LOCAL_INPUT / name
    assert source.exists(), f'{source} is missing: run step 02 first'
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy(source, target)
    print(f'{name}: {target.stat().st_size / 1e6:,.1f} MB')

In [ ]:
NOTEBOOK = 'gpu'
MODELS = ['xgb', 'mlp', 'sage', 'gatv2', 'cheb', 'gin']      # the models trained by this notebook
ONLY_PAIRS = None    # None = every pair; or a list such as ['graph__xgb'] to run only some pairs
RUN_DETERMINISM_CHECK = True   # section 4: train the first triple of each model twice with the same seed

# True = delete the final-run results of this notebook's models on Drive and train them again.
FORCE_RECOMPUTE = False

## 1. Shared code

Data, seeds and metrics (1/4), classical models (2/4), neural models (3/4): identical to the tuning notebooks. Block 4/4 (the tuning driver) is not needed here. Then the final-run code: training of a triple, metrics, files, log.

In [ ]:
# ==== SHARED CODE 1/4 · data, seeds, metrics · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import json
import os
import random
import sys
import time
from datetime import datetime, timezone
from types import SimpleNamespace

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # deterministic cuBLAS: must be set before PyTorch uses the GPU
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

SHARED_CODE_VERSION = 1
CONFIG_VERSION = 1                 # the version of config.json this code was written for
DETERMINISTIC_ALGORITHMS = True    # ask PyTorch for deterministic GPU kernels (only a warning where none exists)


def set_seed(seed):
    """Fix every random generator: Python, NumPy and, when it is loaded, PyTorch (CPU, every GPU, cuDNN)."""
    random.seed(seed)
    np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(DETERMINISTIC_ALGORITHMS, warn_only=True)


def atomic(path, write):
    """Write through a temporary file, then rename it: a run that stops while writing
    never leaves a half-written file on Drive."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    write(tmp)
    os.replace(tmp, path)


def to_builtin(value):
    """Make NumPy numbers JSON-serializable."""
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f'not JSON-serializable: {type(value)}')


def write_csv(df, path):
    atomic(path, lambda tmp: df.to_csv(tmp, index=False, float_format='%.10g'))


def write_json(obj, path):
    text = json.dumps(obj, indent=1, default=to_builtin, ensure_ascii=False)
    atomic(path, lambda tmp: tmp.write_text(text, encoding='utf-8'))


def fmt_time(seconds):
    """Format a duration as '12.3s', '4m 05s' or '1h 07m'."""
    if seconds < 60:
        return f'{seconds:.1f}s'
    minutes, seconds = divmod(int(seconds), 60)
    if minutes < 60:
        return f'{minutes}m {seconds:02d}s'
    hours, minutes = divmod(minutes, 60)
    return f'{hours}h {minutes:02d}m'


def load_inputs(folder):
    """config.json, dataset.parquet and edges.parquet of step 02, with the arrays every later step needs."""
    config = json.loads((folder / 'config.json').read_text(encoding='utf-8'))
    assert config['version'] == CONFIG_VERSION, f"config.json has version {config['version']}, this code expects {CONFIG_VERSION}"
    dataset = pd.read_parquet(folder / 'dataset.parquet')
    edges = pd.read_parquet(folder / 'edges.parquet')
    assert len(dataset) == config['files']['n_rows'] and len(edges) == config['files']['n_edges']
    y = dataset['y'].to_numpy(dtype=np.float64, na_value=np.nan)
    return SimpleNamespace(
        config=config, dataset=dataset,
        split=dataset['split'].to_numpy(), in_eval=dataset['in_eval'].to_numpy(dtype=bool), illicit=(y == 1),
        src=edges['src_row'].to_numpy(np.int64), dst=edges['dst_row'].to_numpy(np.int64))


def scale(values, spec):
    """One column of the profile `scaled` (config['transforms']): z = (t - mean) / std, with
    t = sign(x) log(1 + |x|) for the two log transformations and t = x otherwise (binary: mean 0, std 1)."""
    x = np.asarray(values, dtype=np.float64)
    if spec['transform'] in ('log1p+z', 'signed_log1p+z'):
        x = np.sign(x) * np.log1p(np.abs(x))
    return (x - spec['mean']) / spec['std']


def feature_matrix(data, menu, profile):
    """The input matrix of a menu for every node (float32): raw values (profile `raw`) or transformed
    (profile `scaled`). The menu `none` gives the constant input x = 1."""
    columns = data.config['menus'][menu]['columns']
    if not columns:
        return np.ones((len(data.dataset), 1), dtype=np.float32)
    X = data.dataset[columns].to_numpy(np.float64)
    if profile == 'scaled':
        specs = data.config['transforms']['profiles']['scaled']['columns']
        for k, col in enumerate(columns):
            X[:, k] = scale(X[:, k], specs[col])
    return X.astype(np.float32)


def eval_rows(data, split):
    """Row numbers of the evaluation rows of a split (labeled and with the Elliptic++ data)."""
    return np.flatnonzero((data.split == split) & data.in_eval)


def suggest_params(trial, space):
    """Ask an Optuna trial for one value of every hyper-parameter of a search space of config.json."""
    params = {}
    for name, spec in space.items():
        if spec['type'] == 'categorical':
            params[name] = trial.suggest_categorical(name, spec['choices'])
        elif spec['type'] == 'float':
            params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec['log'])
        elif spec['type'] == 'int':
            params[name] = trial.suggest_int(name, spec['low'], spec['high'], log=spec['log'])
        else:
            raise ValueError(f'unknown type {spec["type"]} for {name}')
    return params

In [ ]:
# ==== SHARED CODE 2/4 · classical models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-cpu, 03-tuning-gpu, and later 04-final-runs-cpu, 04-final-runs-gpu.
# Change every copy together and raise SHARED_CODE_VERSION.
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# XGBoost on the GPU with NumPy inputs prints this at every prediction; it is harmless.
warnings.filterwarnings('ignore', message='.*Falling back to prediction using DMatrix.*')


def fit_classical(model_name, params, fixed, X_train, y_train, X_val, y_val, seed, final=False):
    """Train Logistic Regression, Random Forest or XGBoost on the train rows.
    XGBoost stops adding trees when the validation PR-AUC stops improving (early stopping).
    `final=True` (step 04) also computes the out-of-bag score of the Random Forest."""
    fixed = dict(fixed)
    info = {}
    start = time.perf_counter()
    if model_name == 'logreg':
        model = LogisticRegression(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train)
    elif model_name == 'rf':
        oob = bool(fixed.pop('oob_score_in_final_runs', False)) and final
        model = RandomForestClassifier(**fixed, **params, oob_score=oob, random_state=seed)
        model.fit(X_train, y_train)
        if oob:
            info['oob_average_precision'] = float(average_precision_score(
                y_train, np.nan_to_num(model.oob_decision_function_[:, 1])))
    elif model_name == 'xgb':
        model = XGBClassifier(**fixed, **params, random_state=seed)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        info['best_iteration'] = int(model.best_iteration)
    else:
        raise ValueError(model_name)
    info['seconds'] = time.perf_counter() - start
    scores = model.predict_proba(X_val)[:, 1]
    return SimpleNamespace(model=model, val_scores=scores, val_ap=float(average_precision_score(y_val, scores)), info=info)

In [ ]:
# ==== SHARED CODE 3/4 · neural models · SHARED_CODE_VERSION 1 ====
# Identical copies in: 03-tuning-gpu, and later 04-final-runs-gpu and 05-plots (to recompute embeddings).
# Change every copy together and raise SHARED_CODE_VERSION.
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import ChebConv, GATv2Conv, GINConv, JumpingKnowledge, SAGEConv
from torch_geometric.utils import to_undirected

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def period_data(data, X, split, kind):
    """Tensors of one split on the GPU.
    kind 'graph' (GNNs): every node of the time steps of the split (unknown included), the edges among them made
    undirected; loss and metrics only on the evaluation rows (`mask`).
    kind 'rows' (MLP): only the evaluation rows of the split, no edges."""
    if kind == 'rows':
        rows = eval_rows(data, split)
        y = data.illicit[rows]
        return SimpleNamespace(
            x=torch.tensor(X[rows], device=DEVICE), edge_index=None,
            y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
            mask=torch.ones(len(rows), dtype=torch.bool, device=DEVICE), y_eval=y.astype(int), rows=rows)
    nodes = np.flatnonzero(data.split == split)
    local = np.full(len(data.split), -1)
    local[nodes] = np.arange(len(nodes))
    keep = local[data.src] >= 0
    assert (local[data.dst[keep]] >= 0).all(), 'an edge leaves the split'   # no edge links two time steps
    edge_index = torch.tensor(np.vstack([local[data.src[keep]], local[data.dst[keep]]]), dtype=torch.long)
    mask = data.in_eval[nodes]
    y = data.illicit[nodes]
    return SimpleNamespace(
        x=torch.tensor(X[nodes], device=DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=len(nodes)).to(DEVICE),
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE),
        mask=torch.tensor(mask, device=DEVICE), y_eval=y[mask].astype(int), rows=nodes[mask])


class MLP(nn.Module):
    """`layers` hidden layers of width `hidden` (ReLU, dropout), then one output."""

    def __init__(self, d_in, hidden, layers, dropout):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
        self.out = nn.Linear(hidden, 1)
        self.dropout = dropout

    def forward(self, x, edge_index=None, return_hidden=False):
        for layer in self.hidden:
            x = F.dropout(F.relu(layer(x)), self.dropout, self.training)
        return x if return_hidden else self.out(x).squeeze(-1)


def make_conv(kind, d_in, d_out, args):
    if kind == 'sage':
        return SAGEConv(d_in, d_out, aggr=args['aggr'])
    if kind == 'gatv2':
        heads = args['heads'] if d_out > 1 else 1
        return GATv2Conv(d_in, d_out // heads, heads=heads, dropout=args['att_dropout'])
    if kind == 'cheb':
        return ChebConv(d_in, d_out, K=args['K'], normalization=args.get('normalization', 'sym'))
    if kind == 'gin':
        return GINConv(nn.Sequential(nn.Linear(d_in, d_out), nn.ReLU(), nn.Linear(d_out, d_out)),
                       train_eps=args['train_eps'])
    raise ValueError(kind)


class GNN(nn.Module):
    """`layers` graph convolutions in one of three variants (Ferretti, D'Angelo and Ghini, 2025):
    'base' (the last convolution gives the output), 'lin' (a linear layer after the convolutions),
    'skip' (like 'lin', and the output of the first convolution is added to the output of the last one).
    `jumping_knowledge` ('max' or 'cat', GIN only) combines the outputs of every convolution before the linear layer."""

    def __init__(self, kind, d_in, hidden, layers, variant, dropout, jumping_knowledge=None, **conv_args):
        super().__init__()
        dims = [d_in] + [hidden] * layers
        if variant == 'base':
            dims[-1] = 1
        self.convs = nn.ModuleList(make_conv(kind, a, b, conv_args) for a, b in zip(dims[:-1], dims[1:]))
        self.jk = None if jumping_knowledge is None else JumpingKnowledge(jumping_knowledge)
        width = hidden * layers if jumping_knowledge == 'cat' else hidden
        self.lin = None if variant == 'base' else nn.Linear(width, 1)
        self.variant, self.dropout = variant, dropout

    def forward(self, x, edge_index, return_hidden=False):
        outputs = []
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if self.lin is not None or i < len(self.convs) - 1:
                x = F.dropout(F.relu(x), self.dropout, self.training)
            outputs.append(x)
        if self.variant == 'skip':
            x = x + outputs[0]
        if self.jk is not None:
            x = self.jk(outputs)
        if return_hidden:
            return x
        if self.lin is not None:
            x = self.lin(x)
        return x.squeeze(-1)


def build_neural(model_name, params, d_in, fixed=None):
    """The network of a model from its hyper-parameters (lr and weight_decay belong to the optimizer)."""
    p = {k: v for k, v in params.items() if k not in ('lr', 'weight_decay')}
    fixed = dict(fixed or {})
    if model_name == 'mlp':
        net = MLP(d_in, **p)
    elif model_name == 'gatv2':
        heads, per_head = p.pop('heads'), p.pop('hidden_per_head')
        net = GNN('gatv2', d_in, hidden=heads * per_head, heads=heads, **p)
    elif model_name == 'gin':
        net = GNN('gin', d_in, variant='lin', **p)
    else:
        net = GNN(model_name, d_in, **p, **fixed)
    return net.to(DEVICE)


def predict(net, part):
    """Scores (probability of illicit) of the evaluation rows of one split."""
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(part.x, part.edge_index)[part.mask]).float().cpu().numpy()


def train_neural(net, train, val, lr, weight_decay, pos_weight, max_epochs, patience,
                 report=None, curves=False, train_ap_every=None, keep_best_state=False):
    """Full-batch training with Adam and binary cross-entropy weighted by `pos_weight`.
    After every epoch: PR-AUC on the validation rows; training stops after `patience` epochs without improvement,
    and the result is the best epoch. `report(epoch, best_ap)` is called after every epoch (pruning of the tuning).
    `curves` keeps one row per epoch (train and validation loss, validation PR-AUC, train PR-AUC every
    `train_ap_every` epochs); `keep_best_state` keeps a copy of the weights of the best epoch."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32, device=DEVICE))
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    best_ap, best_epoch, best_state, history, diverged = 0.0, 0, None, [], False
    start = time.perf_counter()
    for epoch in range(1, max_epochs + 1):
        net.train()
        optimizer.zero_grad()
        out = net(train.x, train.edge_index)
        loss = loss_fn(out[train.mask], train.y[train.mask])
        loss.backward()
        optimizer.step()
        net.eval()
        with torch.no_grad():
            val_out = net(val.x, val.edge_index)[val.mask]
        if not torch.isfinite(val_out).all():   # the training diverged (for example, a learning rate too high)
            diverged = True
            break
        val_ap = float(average_precision_score(val.y_eval, val_out.float().cpu().numpy()))
        if curves:
            row = {'epoch': epoch, 'train_loss': float(loss), 'val_loss': float(loss_fn(val_out, val.y[val.mask])),
                   'val_average_precision': val_ap}
            if train_ap_every and epoch % train_ap_every == 0:
                row['train_average_precision'] = float(average_precision_score(train.y_eval, predict(net, train)))
            history.append(row)
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            if keep_best_state:
                best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        if report is not None:
            report(epoch, best_ap)
        if epoch - best_epoch >= patience:
            break
    return {'best_ap': best_ap, 'best_epoch': best_epoch, 'epochs': epoch, 'diverged': diverged,
            'hit_max_epochs': (not diverged) and epoch == max_epochs and epoch - best_epoch < patience,
            'seconds': time.perf_counter() - start, 'n_params': sum(p.numel() for p in net.parameters()),
            'peak_gpu_mb': torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == 'cuda' else None,
            'curves': history, 'best_state': best_state}

In [ ]:
# ==== FINAL-RUN CODE · training, metrics, saving · identical in 04-final-runs-cpu and 04-final-runs-gpu ====
# Change both copies together. It uses the shared code above (blocks 1 and 2, and block 3 for the neural models).
import gc
import math
import platform

import joblib
import sklearn
from sklearn.metrics import precision_recall_curve, roc_auc_score

SPLITS = ('train', 'val', 'test')


def triple_id(menu, model, seed):
    return f'{menu}__{model}__seed{seed}'


def is_neural(config, model):
    return model in config['training']['models']


def free_memory():
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def library_versions():
    import xgboost
    versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__, 'joblib': joblib.__version__}
    if 'torch' in sys.modules:
        import torch
        versions.update(torch=torch.__version__, cuda=torch.version.cuda,
                        gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
    if 'torch_geometric' in sys.modules:
        import torch_geometric
        versions['torch_geometric'] = torch_geometric.__version__
    return versions


def atomic_file(path, write):
    """Like `atomic`, but the temporary file keeps the extension (XGBoost chooses the format from it)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + '.tmp' + path.suffix)
    write(tmp)
    os.replace(tmp, path)


# ---------- metrics ----------
def confusion(y, flagged):
    """tp, fp, tn, fn and the metrics built on them (illicit = positive class)."""
    y, flagged = np.asarray(y).astype(bool), np.asarray(flagged).astype(bool)
    tp, fp = int((flagged & y).sum()), int((flagged & ~y).sum())
    fn, tn = int((~flagged & y).sum()), int((~flagged & ~y).sum())
    ratio = lambda a, b: a / b if b else 0.0
    out = {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
           'precision_illicit': ratio(tp, tp + fp), 'recall_illicit': ratio(tp, tp + fn),
           'f1_illicit': ratio(2 * tp, 2 * tp + fp + fn),
           'precision_licit': ratio(tn, tn + fn), 'recall_licit': ratio(tn, tn + fp),
           'f1_licit': ratio(2 * tn, 2 * tn + fn + fp)}
    out['f1_macro'] = (out['f1_illicit'] + out['f1_licit']) / 2
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    out['mcc'] = (tp * tn - fp * fn) / denominator if denominator else 0.0
    return out


def best_f1_threshold(y, scores):
    """The threshold with the highest illicit F1 (a transaction is flagged when its score >= threshold)."""
    precision, recall, thresholds = precision_recall_curve(y, scores)
    p, r = precision[:-1], recall[:-1]
    f1 = np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-300), 0.0)
    return float(thresholds[int(np.argmax(f1))])


def top_k(y, scores, ks):
    """precision@k and recall@k of the k rows with the highest scores (ties broken by row order)."""
    hits = np.cumsum(np.asarray(y)[np.argsort(-scores, kind='stable')])
    return {label: {'k': int(k), 'precision': float(hits[k - 1] / k), 'recall': float(hits[k - 1] / hits[-1])}
            for label, k in ks.items()}


def bootstrap_weights(n, resamples, seed):
    """How many times each test row appears in each bootstrap resample (one row per resample).
    The same seed and the same row order give the same resamples to every triple: two models can then be
    compared resample by resample."""
    rng = np.random.default_rng(seed)
    picks = rng.integers(0, n, size=(resamples, n))
    flat = (picks + n * np.arange(resamples)[:, None]).ravel()
    return np.bincount(flat, minlength=resamples * n).reshape(resamples, n).astype(np.int32)


def bootstrap_metrics(y, scores, threshold, weights):
    """PR-AUC, ROC-AUC and illicit F1 at `threshold` on every resample, all at once: a resample is a vector of
    weights (how many times each row was drawn). Ties are grouped as in scikit-learn."""
    y = np.asarray(y).astype(bool)
    order = np.argsort(-scores, kind='stable')
    s, ys, w = scores[order], y[order], weights[:, order]
    ends = np.r_[np.flatnonzero(np.diff(s) != 0), len(s) - 1]          # last row of each group of equal scores
    tp = np.cumsum(w * ys, axis=1, dtype=np.int64)[:, ends].astype(np.float64)
    fp = np.cumsum(w * ~ys, axis=1, dtype=np.int64)[:, ends].astype(np.float64)
    recall, fpr = tp / tp[:, -1:], fp / fp[:, -1:]
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    ap = (np.diff(recall, axis=1, prepend=0.0) * precision).sum(axis=1)
    previous = np.hstack([np.zeros((len(w), 1)), recall[:, :-1]])
    roc = (np.diff(fpr, axis=1, prepend=0.0) * (recall + previous) / 2).sum(axis=1)
    flagged = scores >= threshold
    w_tp, w_fp, w_fn = (weights @ (flagged & y).astype(np.int64), weights @ (flagged & ~y).astype(np.int64),
                        weights @ (~flagged & y).astype(np.int64))
    f1 = np.divide(2 * w_tp, 2 * w_tp + w_fp + w_fn, out=np.zeros(len(w)), where=(2 * w_tp + w_fp + w_fn) > 0)
    return pd.DataFrame({'resample': np.arange(len(w)), 'average_precision': ap, 'roc_auc': roc,
                         'f1_illicit_at_val_threshold': f1})


def evaluate(config, y, scores, weights):
    """Every metric of the summary, from labels and scores of the three splits (see the formulas above)."""
    threshold = best_f1_threshold(y['val'], scores['val'])
    thresholds = {'0.5': 0.5, 'best_val_f1': threshold}
    by_split = {}
    for split in SPLITS:
        yy, ss = y[split], scores[split]
        row = {'rows': len(yy), 'illicit': int(yy.sum()), 'prevalence': float(yy.mean()),
               'average_precision': float(average_precision_score(yy, ss)), 'roc_auc': float(roc_auc_score(yy, ss))}
        for name, t in thresholds.items():
            row[f'at_{name}'] = confusion(yy, ss >= t)
        by_split[split] = row
    boot = bootstrap_metrics(y['test'], scores['test'], threshold, weights)
    for b in range(3):   # the fast bootstrap must agree with scikit-learn
        assert abs(boot['average_precision'][b] - average_precision_score(y['test'], scores['test'], sample_weight=weights[b])) < 1e-9
        assert abs(boot['roc_auc'][b] - roc_auc_score(y['test'], scores['test'], sample_weight=weights[b])) < 1e-9
    low, high = config['evaluation']['bootstrap']['interval']
    summary = {col: {'mean': float(boot[col].mean()), 'std': float(boot[col].std(ddof=1)),
                     'low': float(np.percentile(boot[col], low)), 'high': float(np.percentile(boot[col], high))}
               for col in ['average_precision', 'roc_auc', 'f1_illicit_at_val_threshold']}
    metrics = {'by_split': by_split, 'thresholds': thresholds,
               'top_k_test': top_k(y['test'], scores['test'], config['evaluation']['top_k']),
               'bootstrap_test': {'resamples': len(weights), 'interval': [low, high], **summary}}
    return metrics, boot


# ---------- one pair, one triple ----------
def load_best(data, pairs, best_dir):
    """The best hyper-parameters of step 03 for every pair. Stops if a pair is missing, or if it was tuned with another
    version of the shared code or of config.json: the final runs must use exactly the code of the tuning."""
    bests = {}
    for menu, model in pairs:
        path = best_dir / f'{menu}__{model}.json'
        assert path.exists(), f'{path} is missing: finish the tuning of step 03 first'
        best = json.loads(path.read_text(encoding='utf-8'))
        assert best['shared_code_version'] == SHARED_CODE_VERSION, (
            f"{path.name} was tuned with shared code version {best['shared_code_version']}, this notebook has "
            f"{SHARED_CODE_VERSION}: tune that pair again or use the same code")
        assert best['config_version'] == data.config['version'], f'{path.name}: different config.json version'
        bests[(menu, model)] = best
    return bests


def prepare_pair(data, menu, model, best):
    """Inputs of one (menu, model) pair, shared by its three seeds. Classical models: the matrices of the evaluation rows.
    MLP: the evaluation rows as tensors. GNN: the graph of each split (unknown nodes included)."""
    config = data.config
    spec = config['models'][model]
    X = feature_matrix(data, menu, spec['profile'])
    if is_neural(config, model):
        kind = 'rows' if model == 'mlp' else 'graph'
        parts = {split: period_data(data, X, split, kind) for split in SPLITS}
        rows = {split: parts[split].rows for split in SPLITS}
    else:
        rows = {split: eval_rows(data, split) for split in SPLITS}
        parts = {split: X[rows[split]] for split in SPLITS}
    for split in SPLITS:   # every model scores the same rows in the same order (needed by the paired bootstrap)
        assert np.array_equal(rows[split], eval_rows(data, split))
    return SimpleNamespace(menu=menu, model=model, best=best, spec=spec, parts=parts, rows=rows, d_in=X.shape[1],
                           y={split: data.illicit[rows[split]].astype(int) for split in SPLITS},
                           columns=config['menus'][menu]['columns'])


def train_classical_final(config, prep, seed):
    """Logistic Regression, Random Forest or XGBoost, trained from scratch with the tuned hyper-parameters."""
    P, y, model = prep.parts, prep.y, prep.model
    set_seed(seed)
    fit = fit_classical(model, prep.best['params'], prep.best['fixed'], P['train'], y['train'], P['val'], y['val'],
                        seed, final=True)
    m = fit.model
    start = time.perf_counter()
    scores = {split: m.predict_proba(P[split])[:, 1].astype(np.float64) for split in SPLITS}
    assert np.allclose(scores['val'], fit.val_scores)
    info = {**fit.info, 'predict_seconds': time.perf_counter() - start}
    extra, curves, importances = {}, None, None
    if model == 'logreg':
        importances = pd.DataFrame({'feature': prep.columns, 'importance_type': 'coefficient (z-scored input)',
                                    'value': m.coef_[0]})
        info['intercept'] = float(m.intercept_[0])
    elif model == 'rf':
        importances = pd.DataFrame({'feature': prep.columns, 'importance_type': 'mean decrease in impurity',
                                    'value': m.feature_importances_})
        if hasattr(m, 'oob_decision_function_'):   # NaN for a row that was in every bootstrap sample
            extra['oob_score'] = {'train': m.oob_decision_function_[:, 1]}
    elif model == 'xgb':
        gain = m.get_booster().get_score(importance_type='gain')
        importances = pd.DataFrame({'feature': prep.columns, 'importance_type': 'gain',
                                    'value': [gain.get(f'f{k}', 0.0) for k in range(len(prep.columns))]})
        history = m.evals_result()['validation_0']
        n_trees = len(next(iter(history.values())))
        curves = pd.DataFrame({'tree': np.arange(1, n_trees + 1),
                               **{f'val_{name}_xgboost': values for name, values in history.items()}})
        every = config['final_runs']['train_average_precision_every']
        for k in range(every, n_trees + 1, every):
            for split in ('train', 'val'):
                part_scores = m.predict_proba(P[split], iteration_range=(0, k))[:, 1]
                curves.loc[k - 1, f'{split}_average_precision'] = average_precision_score(y[split], part_scores)
        info['n_trees'] = n_trees
    return SimpleNamespace(scores=scores, info=info, extra=extra, curves=curves, importances=importances, model=m)


def neural_scores(net, part):
    """Logits and scores of the evaluation rows, in float64. The float32 sigmoid of `predict` rounds every logit
    above about 17 to exactly 1.0: the most confident transactions would become ties, and ties change the PR-AUC."""
    import torch
    net.eval()
    with torch.no_grad():
        logits = net(part.x, part.edge_index)[part.mask].double().cpu().numpy()
    with np.errstate(over='ignore'):
        return logits, 1 / (1 + np.exp(-logits))


def train_neural_final(config, prep, seed):
    """MLP or GNN trained from scratch with the tuned hyper-parameters; the result is the network of the best epoch."""
    P, params, best = prep.parts, prep.best['params'], prep.best
    set_seed(seed)
    net = build_neural(prep.model, params, prep.d_in, best['fixed'])
    result = train_neural(net, P['train'], P['val'], params['lr'], params['weight_decay'], config['labels']['pos_weight'],
                          best['max_epochs'], best['patience'], curves=True,
                          train_ap_every=config['final_runs']['train_average_precision_every'], keep_best_state=True)
    assert result['best_state'] is not None, 'no epoch improved the validation PR-AUC (the training diverged at once)'
    net.load_state_dict(result['best_state'])
    start = time.perf_counter()
    logits, scores = {}, {}
    for split in SPLITS:
        logits[split], scores[split] = neural_scores(net, P[split])
    info = {k: result[k] for k in ['best_epoch', 'epochs', 'hit_max_epochs', 'diverged', 'seconds', 'n_params', 'peak_gpu_mb']}
    info['predict_seconds'] = time.perf_counter() - start
    # the reloaded best epoch gives back its validation PR-AUC (up to GPU rounding)
    info['val_check_difference'] = abs(float(average_precision_score(prep.y['val'], scores['val'])) - result['best_ap'])
    assert info['val_check_difference'] < 1e-3, info['val_check_difference']
    state = {k: v.detach().cpu() for k, v in result['best_state'].items()}
    return SimpleNamespace(scores=scores, info=info, extra={'logit': logits}, curves=pd.DataFrame(result['curves']),
                           importances=None,
                           model=state)


def save_model(model_name, model, stem):
    """joblib (compressed) for LogReg and RF, XGBoost's own JSON, PyTorch state_dict for the networks."""
    if model_name in ('logreg', 'rf'):
        path = stem.with_name(stem.name + '.joblib')
        atomic_file(path, lambda tmp: joblib.dump(model, tmp, compress=3))
    elif model_name == 'xgb':
        path = stem.with_name(stem.name + '.json')
        atomic_file(path, lambda tmp: model.save_model(str(tmp)))
    else:
        import torch
        path = stem.with_name(stem.name + '.pt')
        atomic_file(path, lambda tmp: torch.save(model, tmp))
    return path


def predictions_table(data, prep, scores, extra):
    """One row per evaluation row of train, val and test: row number, txId, time step, split, label, score."""
    tables = []
    for split in SPLITS:
        rows = prep.rows[split]
        table = pd.DataFrame({'row': rows, 'txId': data.dataset['txId'].to_numpy()[rows],
                              'time_step': data.dataset['time_step'].to_numpy()[rows], 'split': split,
                              'y': prep.y[split], 'score': scores[split]})
        for column, by_split in extra.items():   # 'logit' (networks), 'oob_score' (Random Forest, train rows)
            table[column] = by_split.get(split, np.nan)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


def run_triple(data, prep, seed, dirs, weights, save=True):
    """Train one triple (menu, model, seed). With save=True, write its six files; the summary is written last,
    so its presence means "triple finished"."""
    config, model = data.config, prep.model
    start = time.perf_counter()
    run = train_neural_final(config, prep, seed) if is_neural(config, model) else train_classical_final(config, prep, seed)
    if not save:
        return None, run
    metrics, boot = evaluate(config, prep.y, run.scores, weights)
    tid = triple_id(prep.menu, model, seed)
    files = {'predictions': f'predictions/{tid}.parquet', 'bootstrap': f'bootstrap/{tid}.csv'}
    # Parquet keeps the exact float64 scores: a CSV with 10 digits would turn the most confident ones into ties
    table = predictions_table(data, prep, run.scores, run.extra)
    atomic(dirs.predictions / f'{tid}.parquet', lambda tmp: table.to_parquet(tmp, index=False))
    write_csv(boot, dirs.bootstrap / f'{tid}.csv')
    if run.curves is not None:
        write_csv(run.curves, dirs.curves / f'{tid}.csv')
        files['curves'] = f'curves/{tid}.csv'
    if run.importances is not None:
        write_csv(run.importances, dirs.importances / f'{tid}.csv')
        files['importances'] = f'importances/{tid}.csv'
    model_path = save_model(model, run.model, dirs.models / tid)
    files['model'] = f'models/{model_path.name}'
    best = prep.best
    record = {
        'triple': tid, 'menu': prep.menu, 'model': model, 'seed': seed, 'notebook': NOTEBOOK,
        'n_columns': len(prep.columns), 'd_in': prep.d_in, 'profile': prep.spec['profile'],
        'params': best['params'], 'fixed': best['fixed'], 'max_epochs': best['max_epochs'], 'patience': best['patience'],
        'val_average_precision_tuning': best['val_average_precision'],
        'tuning': {'best_trial': best['best_trial'], 'study': best['study'], 'tuning_seed': best['tuning_seed']},
        'training': run.info, 'metrics': metrics,
        'files': files, 'model_mb': model_path.stat().st_size / 1e6,
        'seconds_total': time.perf_counter() - start,
        'shared_code_version': SHARED_CODE_VERSION, 'config_version': config['version'], 'versions': VERSIONS,
        'created': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    }
    write_json(record, dirs.summary / f'{tid}.json')
    return record, run


# ---------- driver ----------
class Progress:
    """Log lines with triples done and left, time spent and an estimate of the time left (expected cost of each
    triple from the tuning, corrected by the speed measured in this session). Lines are also appended to a log file."""

    def __init__(self, expected, done, log_path):
        self.expected, self.done, self.log_path = expected, set(done), log_path
        self.work, self.start, self.lines = 0.0, time.time(), []

    def finish(self, tid, cost=None):
        self.done.add(tid)
        self.work += self.expected[tid] if cost is None else cost

    def status(self):
        elapsed = time.time() - self.start
        speed = elapsed / self.work if self.work > 0 else 1.0
        left = sum(s for t, s in self.expected.items() if t not in self.done) * speed
        return (f'triples done {len(self.done)}, left {len(self.expected) - len(self.done)} · '
                f'elapsed {fmt_time(elapsed)}, left ~{fmt_time(left)}')

    def print(self, line):
        print(line, flush=True)
        self.lines.append(f'{datetime.now():%Y-%m-%d %H:%M:%S} {line}')

    def flush(self):
        if self.lines and self.log_path is not None:
            self.log_path.parent.mkdir(parents=True, exist_ok=True)
            with open(self.log_path, 'a', encoding='utf-8') as f:
                f.write('\n'.join(self.lines) + '\n')
        self.lines = []


def expected_seconds(model, best):
    """Expected seconds of one final run: the training time of the best trial of step 03, plus the extra work of
    step 04 (out-of-bag score and compressed file for the RF, curves for XGBoost, metrics and files for everyone)."""
    base = best['best_trial_info'].get('seconds', 1.0)
    return base * {'rf': 1.3, 'logreg': 1.0, 'xgb': 1.0}.get(model, 1.1) + {'rf': 15, 'xgb': 15}.get(model, 4)


def plan_final_runs(data, models, dirs, only_pairs=None):
    """Every triple of this notebook, the tuned hyper-parameters, and what is already on Drive."""
    config = data.config
    pairs = [(menu, model) for menu, model in config['pairs']
             if model in models and (only_pairs is None or f'{menu}__{model}' in only_pairs)]
    bests = load_best(data, pairs, dirs.best)
    seeds = config['final_runs']['seeds']
    expected = {triple_id(m, mo, s): expected_seconds(mo, bests[(m, mo)]) for m, mo in pairs for s in seeds}
    done = [t for t in expected if (dirs.summary / f'{t}.json').exists()]
    progress = Progress(expected, done, dirs.logs / f'04-final-runs-{NOTEBOOK}.log')
    progress.print(f'{len(expected)} triples in this notebook ({len(pairs)} pairs x {len(seeds)} seeds), '
                   f'{len(done)} already done. Expected time of the missing ones: '
                   f'~{fmt_time(sum(s for t, s in expected.items() if t not in done))}.')
    progress.flush()
    return SimpleNamespace(pairs=pairs, bests=bests, seeds=seeds, progress=progress)


def triple_line(label, record, seconds):
    m = record['metrics']['by_split']
    return (f"{label} val PR-AUC {m['val']['average_precision']:.4f} (tuning {record['val_average_precision_tuning']:.4f}) · "
            f"test PR-AUC {m['test']['average_precision']:.4f} · {fmt_time(seconds)}")


def run_final_runs(data, plan, dirs, weights):
    """Train every missing triple, pair by pair (the inputs of a pair are prepared once for its three seeds)."""
    progress, n = plan.progress, len(plan.progress.expected)
    k = 0
    for menu, model in plan.pairs:
        todo = [s for s in plan.seeds if triple_id(menu, model, s) not in progress.done]
        if not todo:
            k += len(plan.seeds)
            progress.print(f'[pair · {model} × {menu}] all {len(plan.seeds)} seeds already done')
            continue
        prep = prepare_pair(data, menu, model, plan.bests[(menu, model)])
        for seed in plan.seeds:
            k += 1
            tid = triple_id(menu, model, seed)
            if seed not in todo:
                continue
            label = f'[triple {k}/{n} · {model} × {menu} · seed {seed}]'
            start = time.time()
            record, _ = run_triple(data, prep, seed, dirs, weights)
            progress.finish(tid)
            progress.print(f'{triple_line(label, record, time.time() - start)} · {progress.status()}')
            progress.flush()
            free_memory()
        del prep
        free_memory()
    progress.flush()


def determinism_check(data, plan, dirs, weights, path):
    """For each model: train the first triple twice with the same seed and measure how much the scores differ.
    The first training is the real triple (saved); if it was saved in an earlier session, its saved scores
    are the reference. One row per model in `path`; models already in it are skipped."""
    config, progress = data.config, plan.progress
    rows = pd.read_csv(path).to_dict('records') if path.exists() else []
    checked = {r['model'] for r in rows if r['check'] == 'same seed twice'}
    for model in dict.fromkeys(mo for _, mo in plan.pairs):
        if model in checked:
            progress.print(f'[check · {model}] already in {path.name}')
            continue
        menu = next(m for m, mo in plan.pairs if mo == model)
        seed = plan.seeds[0]
        tid = triple_id(menu, model, seed)
        prep = prepare_pair(data, menu, model, plan.bests[(menu, model)])
        start = time.time()
        if tid in progress.done:
            saved = pd.read_parquet(dirs.predictions / f'{tid}.parquet')
            first = {split: saved.loc[saved['split'] == split, 'score'].to_numpy() for split in SPLITS}
            first_info, reference = json.loads((dirs.summary / f'{tid}.json').read_text(encoding='utf-8'))['training'], \
                'saved scores'
        else:
            record, run = run_triple(data, prep, seed, dirs, weights)
            progress.finish(tid)
            progress.print(f"{triple_line(f'[check · {model} × {menu} · seed {seed}] run 1:', record, time.time() - start)}")
            first, first_info, reference = run.scores, run.info, 'first run, in memory'
        _, again = run_triple(data, prep, seed, dirs, weights, save=False)
        progress.work += progress.expected[tid]
        diff = max(float(np.max(np.abs(first[split] - again.scores[split]))) for split in SPLITS)
        row = {'model': model, 'menu': menu, 'seed': seed, 'check': 'same seed twice', 'reference': reference,
               'max_abs_score_difference': diff}
        for split in ('val', 'test'):
            a = average_precision_score(prep.y[split], first[split])
            b = average_precision_score(prep.y[split], again.scores[split])
            row.update({f'{split}_average_precision_run1': a, f'{split}_average_precision_run2': b,
                        f'{split}_average_precision_difference': abs(a - b)})
        for key in ['best_epoch', 'best_iteration']:
            if key in again.info:
                row[f'{key}_run1'], row[f'{key}_run2'] = first_info.get(key), again.info[key]
        rows.append(row)
        write_csv(pd.DataFrame(rows), path)
        progress.print(f"[check · {model} × {menu} · seed {seed}] same seed twice: largest score difference {diff:.2e}, "
                       f"test PR-AUC difference {row['test_average_precision_difference']:.2e} · {progress.status()}")
        progress.flush()
        del prep, again
        free_memory()
    return pd.DataFrame(rows)


def flat_row(r):
    """One row of runs.csv from a summary."""
    m = r['metrics']
    row = {'menu': r['menu'], 'model': r['model'], 'seed': r['seed'], 'n_columns': r['n_columns'],
           'val_average_precision_tuning': r['val_average_precision_tuning']}
    for split in SPLITS:
        row[f'{split}_average_precision'] = m['by_split'][split]['average_precision']
        row[f'{split}_roc_auc'] = m['by_split'][split]['roc_auc']
    row['train_oob_average_precision'] = r['training'].get('oob_average_precision')
    row['threshold_best_val_f1'] = m['thresholds']['best_val_f1']
    for name in ['0.5', 'best_val_f1']:
        for split in ('val', 'test'):
            c = m['by_split'][split][f'at_{name}']
            for key in ['f1_illicit', 'precision_illicit', 'recall_illicit', 'f1_macro', 'mcc']:
                row[f'{split}_{key}_at_{name}'] = c[key]
    for key in ['average_precision', 'roc_auc', 'f1_illicit_at_val_threshold']:
        b = m['bootstrap_test'][key]
        row[f'test_{key}_bootstrap_low'], row[f'test_{key}_bootstrap_high'] = b['low'], b['high']
    for label, t in m['top_k_test'].items():
        row[f'test_precision_at_{t["k"]}'], row[f'test_recall_at_{t["k"]}'] = t['precision'], t['recall']
    info = r['training']
    for key in ['best_epoch', 'epochs', 'hit_max_epochs', 'best_iteration', 'n_trees', 'seconds', 'peak_gpu_mb', 'n_params']:
        row[key] = info.get(key)
    row.update(model_mb=r['model_mb'], seconds_total=r['seconds_total'], notebook=r['notebook'],
               shared_code_version=r['shared_code_version'])
    return row


def write_runs(dirs, path):
    """runs.csv: one row per finished triple, of both final-run notebooks."""
    runs = pd.DataFrame([flat_row(json.loads(f.read_text(encoding='utf-8'))) for f in sorted(dirs.summary.glob('*.json'))])
    if len(runs):
        write_csv(runs, path)
    return runs

## 2. Load the inputs

`config.json` (step 02) decides menus, transformations, seeds and metrics; `03-tuning/best/` has the hyper-parameters. The checks: the files of step 02 have the expected version and sizes, every pair of this notebook was tuned with the same shared code, and a GPU is available. The bootstrap resamples are drawn once here, so they are the same for every triple.

In [ ]:
DIRS = SimpleNamespace(best=TUNING_DIR / 'best', predictions=RUNS_DIR / 'predictions', summary=RUNS_DIR / 'summary',
                       curves=RUNS_DIR / 'curves', importances=RUNS_DIR / 'importances', bootstrap=RUNS_DIR / 'bootstrap',
                       models=RUNS_DIR / 'models', logs=RUNS_DIR / 'logs')
start = time.time()
data = load_inputs(LOCAL_INPUT)
print(f'Inputs loaded in {fmt_time(time.time() - start)}: {len(data.dataset):,} rows, config version {data.config["version"]}, '
      f'final seeds {data.config["final_runs"]["seeds"]}.')
for split in SPLITS:
    rows = eval_rows(data, split)
    print(f'{split}: {len(rows):,} evaluation rows, {data.illicit[rows].sum():,} illicit')

assert DEVICE.type == 'cuda', 'No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run again.'
print('GPU:', torch.cuda.get_device_name(0))

VERSIONS = library_versions()
boot = data.config['evaluation']['bootstrap']
assert boot['replace'], 'the bootstrap of this notebook draws with replacement'
WEIGHTS = bootstrap_weights(len(eval_rows(data, 'test')), boot['resamples'], boot['seed'])
DETERMINISM_FILE = RUNS_DIR / f'determinism-{NOTEBOOK}.csv'

if FORCE_RECOMPUTE:
    for menu, model in data.config['pairs']:
        if model in MODELS:
            for folder in ['predictions', 'summary', 'curves', 'importances', 'bootstrap', 'models']:
                for path in (RUNS_DIR / folder).glob(f'{menu}__{model}__seed*'):
                    path.unlink()
    DETERMINISM_FILE.unlink(missing_ok=True)

plan = plan_final_runs(data, MODELS, DIRS, ONLY_PAIRS)

## 3. Tests on toy problems

Before the real runs, quick checks with known answers:
- **metrics**: confusion counts, F1, MCC, best threshold and top-k on a hand-made example; the fast bootstrap (all 1,000 resamples at once) gives exactly the values of scikit-learn, also with tied scores;
- **models**: XGBoost learn a toy table, the same seed gives the same scores, and the saved file gives back the same scores;
- **networks**: MLP and GraphSAGE on a toy graph: the weights of the best epoch are restored, and the saved `state_dict` rebuilds a network with the same scores.

In [ ]:
# Metrics on a hand-made example with known answers.
y_toy = np.array([1, 1, 0, 0, 0, 1, 0, 0])
s_toy = np.array([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2])
c = confusion(y_toy, s_toy >= 0.5)            # flagged: the first five
assert (c['tp'], c['fp'], c['tn'], c['fn']) == (2, 3, 2, 1), c
assert abs(c['f1_illicit'] - 0.5) < 1e-12 and abs(c['mcc'] - 1 / 15) < 1e-12, c
assert best_f1_threshold(y_toy, s_toy) == 0.8   # flag 0.9 and 0.8: precision 1, recall 2/3, F1 0.8
t = top_k(y_toy, s_toy, {'two': 2, 'four': 4})
assert t['two']['precision'] == 1.0 and abs(t['four']['recall'] - 2 / 3) < 1e-12, t

# The fast bootstrap against scikit-learn, on data with many ties (scores rounded to one decimal).
rng = np.random.default_rng(1)
y_b = (rng.random(500) < 0.1).astype(int)
s_b = np.round(np.clip(0.3 * y_b + rng.random(500) * 0.7, 0, 1), 1)
w_b = bootstrap_weights(500, 20, seed=0)
assert (w_b.sum(axis=1) == 500).all() and np.array_equal(w_b, bootstrap_weights(500, 20, seed=0))
boot_b = bootstrap_metrics(y_b, s_b, 0.5, w_b)
for b in range(20):
    assert abs(boot_b['average_precision'][b] - average_precision_score(y_b, s_b, sample_weight=w_b[b])) < 1e-9
    assert abs(boot_b['roc_auc'][b] - roc_auc_score(y_b, s_b, sample_weight=w_b[b])) < 1e-9
    repeated = np.repeat(np.arange(500), w_b[b])
    assert abs(boot_b['f1_illicit_at_val_threshold'][b] - confusion(y_b[repeated], s_b[repeated] >= 0.5)['f1_illicit']) < 1e-12
print('Metrics test passed: confusion, threshold, top-k, and the fast bootstrap equals scikit-learn on 20 resamples.')

In [ ]:
# Every model of this notebook: same seed -> same scores, and the saved file gives back the same scores.
import tempfile

toy_dir = Path(tempfile.mkdtemp())
rng = np.random.default_rng(0)
toy_y = (rng.random(1000) < 0.1).astype(int)
toy_X = np.column_stack([toy_y + 0.3 * rng.standard_normal(1000), rng.standard_normal(1000)]).astype(np.float32)
tr, va = np.arange(700), np.arange(700, 1000)
toy_params = {'xgb': {'learning_rate': 0.1, 'max_depth': 3, 'min_child_weight': 1.0, 'subsample': 0.8, 'colsample_bytree': 1.0, 'reg_lambda': 1.0, 'gamma': 0.0, 'scale_pos_weight': 1.0}}
for model, params in toy_params.items():
    fixed = data.config['models'][model].get('fixed', {})
    fits = []
    for _ in range(2):
        set_seed(0)
        fits.append(fit_classical(model, params, fixed, toy_X[tr], toy_y[tr], toy_X[va], toy_y[va], seed=0, final=True))
    fit = fits[0]
    assert fit.val_ap > 0.7, (model, fit.val_ap)
    assert np.allclose(fits[0].val_scores, fits[1].val_scores, atol=1e-6), f'{model}: same seed, different scores'
    path = save_model(model, fit.model, toy_dir / model)
    loaded = XGBClassifier()
    loaded.load_model(str(path))
    best_trees = (0, int(fit.model.best_iteration) + 1)   # the trees up to the best iteration (early stopping)
    again = loaded.predict_proba(toy_X[va], iteration_range=best_trees)[:, 1]
    assert np.allclose(again, fit.model.predict_proba(toy_X[va], iteration_range=best_trees)[:, 1], atol=1e-6), model
    print(f'{model}: toy validation PR-AUC {fit.val_ap:.3f}, reproducible, saved and reloaded ({path.stat().st_size / 1e3:.0f} KB)')

In [ ]:
# MLP and GraphSAGE on a toy graph of pairs u - v, where the label of v depends only on its partner u:
# the best epoch is restored, and the saved state_dict rebuilds a network with the same scores.
def toy_graph(seed, n=600):
    rng = np.random.default_rng(seed)
    f_u, f_v = rng.standard_normal(n), rng.standard_normal(n)
    y = np.concatenate([np.zeros(n), (f_u > 1).astype(float)])
    mask = np.concatenate([np.zeros(n, dtype=bool), np.ones(n, dtype=bool)])
    edge_index = torch.tensor(np.vstack([np.arange(n), np.arange(n) + n]), dtype=torch.long)
    graph = SimpleNamespace(
        x=torch.tensor(np.concatenate([f_u, f_v])[:, None], dtype=torch.float32, device=DEVICE),
        edge_index=to_undirected(edge_index, num_nodes=2 * n).to(DEVICE),
        y=torch.tensor(y, dtype=torch.float32, device=DEVICE), mask=torch.tensor(mask, device=DEVICE),
        y_eval=y[mask].astype(int))
    rows = SimpleNamespace(x=graph.x[mask], edge_index=None, y=graph.y[mask],
                           mask=torch.ones(n, dtype=torch.bool, device=DEVICE), y_eval=graph.y_eval)
    return graph, rows


(toy_train_graph, toy_train_rows), (toy_val_graph, toy_val_rows) = toy_graph(0), toy_graph(1)
toy_pos_weight = float((toy_train_graph.y_eval == 0).sum() / toy_train_graph.y_eval.sum())
for model, params, train_part, val_part in [
        ('mlp', {'layers': 2, 'hidden': 32, 'dropout': 0.1}, toy_train_rows, toy_val_rows),
        ('sage', {'layers': 2, 'hidden': 32, 'aggr': 'mean', 'variant': 'lin', 'dropout': 0.1}, toy_train_graph, toy_val_graph)]:
    set_seed(0)
    net = build_neural(model, params, 1)
    result = train_neural(net, train_part, val_part, 1e-2, 0.0, toy_pos_weight, 200, 50,
                          curves=True, train_ap_every=10, keep_best_state=True)
    net.load_state_dict(result['best_state'])
    restored = neural_scores(net, val_part)[1]
    assert abs(average_precision_score(val_part.y_eval, restored) - result['best_ap']) < 1e-4, model
    assert 'train_average_precision' in result['curves'][9], model
    path = save_model(model, {k: v.detach().cpu() for k, v in result['best_state'].items()}, toy_dir / model)
    rebuilt = build_neural(model, params, 1)
    rebuilt.load_state_dict(torch.load(path, map_location=DEVICE))
    assert np.allclose(neural_scores(rebuilt, val_part)[1], restored, atol=1e-6), model
    print(f'{model}: toy best epoch {result["best_epoch"]} restored (PR-AUC {result["best_ap"]:.3f}), saved and rebuilt')

## 4. Reproducibility check

For each model, the first triple (first menu, seed 0) is trained **twice with the same seed**, and the largest difference between the two sets of scores is written to `determinism-gpu.csv`, with the two PR-AUCs. On the GPU some PyTorch and PyG operations add numbers in the order the threads finish, so two runs can differ by about $10^{-6}$, and the difference can grow during training (and change the best epoch). This check measures it, for the report. The first of the two trainings is the real triple, so the check costs one extra training per model (a few minutes in all).

In [ ]:
if RUN_DETERMINISM_CHECK:
    determinism = determinism_check(data, plan, DIRS, WEIGHTS, DETERMINISM_FILE)
    display(determinism)

## 5. Final runs

One log line per triple, for example

```
[triple 14/96 · sage × local_agg · seed 1] val PR-AUC 0.9102 (tuning 0.9096) · test PR-AUC 0.7013 · 1m 05s · triples done 14, left 82 · elapsed 20m 10s, left ~1h 05m
```

The validation PR-AUC of the final run is usually a little below that of the tuning, which is the best of many trials. The estimate of the time left is weighted by the training time of each pair in the tuning and corrected by the speed measured in this session.

In [ ]:
run_final_runs(data, plan, DIRS, WEIGHTS)

## 6. Summary

`runs.csv` has one row per finished triple, of both final-run notebooks (the columns are listed in `flat_row` above). The table below shows the **test PR-AUC, mean ± standard deviation over the three seeds** (rows: models, columns: menus). It is a quick look: the analysis and the plots are in step 05.

In [ ]:
runs = write_runs(DIRS, RUNS_DIR / 'runs.csv')
mine = [triple_id(m, mo, s) for m, mo in plan.pairs for s in plan.seeds]
done = [t for t in mine if (DIRS.summary / f'{t}.json').exists()]
print(f'Triples of this notebook finished: {len(done)} of {len(mine)}.')
assert len(done) == len(mine), 'some triples are missing: run the notebook again'
if len(runs):
    stats = runs.groupby(['model', 'menu'])['test_average_precision'].agg(['mean', 'std']).reset_index()
    stats['cell'] = stats.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)
    table = stats.pivot(index='model', columns='menu', values='cell')
    display(table.reindex(columns=[m for m in data.config['menus'] if m in table.columns]).fillna(''))

In [ ]:
drive.flush_and_unmount()
print('Every file is on Drive in', RUNS_DIR)